# Marine Accident Risk — 1) End-to-End 파이프라인 데모

본 노트북은 **회사 NDA 로 인해 본인이 별도로 재현한 데모**입니다.
데이터는 모두 공개 OPEN API + 공개 격자 + 공개 사고 통계만 사용합니다.

흐름:
1. 사고 xlsx + level4 격자 로드 → bbox 필터
2. (선택) NMPNT 측위정보원 1시간 평균 기상 결합
3. positive(사고) + negative 샘플링 → LightGBM 학습 (5-fold CV)
4. SHAP 분해
5. 격자×시간 추론 (FastAPI / Streamlit 와 동일한 구조)

In [ ]:
import os, pandas as pd
from marine_accident_risk.config import Config
from marine_accident_risk.data.accidents import load_accidents
from marine_accident_risk.data.grid import load_grid, filter_grid
from marine_accident_risk.features.build import build_dataset
from marine_accident_risk.models import lgbm
from marine_accident_risk.models.shap_analyzer import explain

cfg = Config.load('../configs/default.yaml')
cfg.raw['paths']['accidents_xlsx'] = '../data/raw/accidents.xlsx'
cfg.raw['paths']['grid_csv'] = '../data/raw/level4.csv'
accidents = load_accidents(cfg.get('paths','accidents_xlsx'))
grid = load_grid(cfg.get('paths','grid_csv'))
print('accidents', accidents.shape, 'grid', grid.shape)
print('bbox', cfg.bbox)

In [ ]:
# 2) 학습 데이터셋 (기상 없이 격자/시간만 — 키 있으면 weather parquet 결합)
bundle = build_dataset(cfg, accidents, grid, pd.DataFrame())
print('rows', len(bundle.df), 'feats', len(bundle.feature_cols))
bundle.df.head()

In [ ]:
# 3) LightGBM 5-fold CV
params = cfg.get('model','params')
res = lgbm.train_cv(bundle.df, bundle.feature_cols, 'y',
                    params=params,
                    n_splits=int(cfg.get('model','cv','n_splits')),
                    early_stopping_rounds=int(cfg.get('model','cv','early_stopping_rounds')),
                    n_estimators=int(params.get('n_estimators',400)))
print('OOF AUC', res.oof_auc, 'PR_AUC', res.oof_pr_auc)
for fm in res.fold_metrics: print(fm)

In [ ]:
# 4) SHAP
sv = explain(res.booster, bundle.df[bundle.feature_cols])
mean_abs = sv.drop(columns=['__base_value__']).abs().mean().sort_values(ascending=False)
mean_abs.head(10)

In [ ]:
# 5) 단일 격자×시간 추론 (FastAPI 의 /predict 와 동일 흐름)
from marine_accident_risk.features.build import _add_time_features
import numpy as np
g = filter_grid(grid, cfg.bbox)
cell = g.iloc[0]
row = pd.DataFrame([{
    'gid': int(cell['gid']),
    'ts': pd.Timestamp('2024-06-15 14:00'),
    'lat_center': float(cell['lat_center']),
    'lon_center': float(cell['lon_center']),
    'gid_acc_log1p': float(np.log1p(
        (accidents['gid'] == cell['gid']).sum() if 'gid' in accidents.columns else 0)),
    'hour_dow_acc_log1p': 0.0,
    'cell_area_deg2': 0.025**2,
}])
row = pd.concat([row, _add_time_features(row['ts'])], axis=1)
for k in ('wind_speed','air_temperature','air_pressure','humidity','water_temper','horizon_visibl'):
    row[f'wx_{k}'] = np.nan
proba = lgbm.predict_proba(res.booster, row[bundle.feature_cols])
print('grid', cell['gid'], '@2024-06-15 14:00 -> p=', float(proba[0]))